# Setup

In [1]:
import os
import sys

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname("."), "src")))
import pandas as pd
from processor.main import Processor
from processor.table.representation.impl.df_table import DFTable
from processor.table.store.table_store_factory import ImplementedTableStore

In [2]:
# OpenAI model
# from dotenv import load_dotenv
# load_dotenv()
# api_key = os.getenv('OPENAI_API_KEY')
# model = 'gpt-4o-mini-2024-07-18'

# Local Model
model = "src/processor/model/weight/qwen25-7b"

embed_path = "src/processor/model/weight/bge-base"
processor = Processor(
    model, embed_path, ImplementedTableStore.PY_TABLE_STORE, "processor_output"
)

In [3]:
# Define constants
DB_SCHEMA = "E2E_SCHEMA"
DB_SCHEMA_AFTER_UNION = DB_SCHEMA + "_AFTER_UNION"
QUESTION_1 = "Assuming all UPS ground shipments are delayed by 3 days before they are shipped (i.e., ship 3 days later than scheduled), how many items will be impacted?"

In [4]:
# Prepare metadata (table descriptions)
metadata = pd.read_csv("data_src/buysite/metadata.csv")
table_descriptions: dict[str, str] = dict()
for i, row in metadata.iterrows():
    table_descriptions[row["table"]] = row["value"]

## Can be skipped if alread indexed

In [5]:
# # Index the tables, retrieved by Pneuma for QUESTION_1
# from tqdm import tqdm
# import duckdb
# try:
#     processor.ctx.table_store.create_db_schema(DB_SCHEMA)
# except:
#     pass

# TABLE_PATH_PREFIX = "data_src/buysite/dataset"
# retrieved_tables_q1 = [
#     "JI_ASN", "JI_ASN_CARRIER", "JI_ASN_LINE", "JI_PURCHASE_ORDER_LINE", "JI_FULFILLMENT_CENTER_TERMS_CONDITIONS"
# ]
# for table_name in tqdm(retrieved_tables_q1):
#     table_path = os.path.join(TABLE_PATH_PREFIX, f"{table_name}.csv")
#     query = f"""
#     SELECT * FROM read_csv_auto('{table_path}')
#     LIMIT 1000"""
#     df = duckdb.query(query).to_df()
#     processor.ctx.table_store.add_table(
#         DB_SCHEMA,
#         table_name,
#         DFTable(df),
#         False,
#         False,
#     )
# processor.ctx.table_store.checkpoint()

# Step 1: Schema Enhancement & Target Schema Generation

In [5]:
QUESTION_1

'Assuming all UPS ground shipments are delayed by 3 days before they are shipped (i.e., ship 3 days later than scheduled), how many items will be impacted?'

In [ ]:
schema_generator_system_prompt = """You are an expert in data integration. Your task is to:

1. Determine the target schema: the set of necessary columns required to directly answer a given question, without needing joins, external lookups, or multiple steps (e.g., averaging or aggregating across different tables).
- The first column must always be 'ID', which is the primary key of the table.
- The target schema must be self-sufficient: it must include all attributes needed to answer the question, filter relevant entities, and compute the result directly.
- Always include **any entity or attribute mentioned in the question** that would be used for filtering, grouping, or comparison—this includes product names, customer types, service providers (e.g., "UPS Ground Service"), shipping modes, regions, dates, and so on.
- If the question involves counting items, quantities, or totals, ensure that the schema includes any necessary numeric fields (e.g., quantity per shipment) to support accurate computation.
- If thresholds are given (e.g., delays by a certain number of days), use >= instead of > unless the question specifically states "more than" or "strictly greater than".
- Use exact matching for entity names. Do not simplify or generalize (e.g., retain "Amazon Ground Delivery" instead of shortening to "Amazon").

2. Simulate the SQL query that would answer the question using only the columns in the target schema.
- Assume the table is named "target_table".
- Your SQL must be executable without relying on missing columns or external logic.
- Use ANSI-standard SQL syntax. Avoid engine-specific functions (e.g., don't use MySQL's DATE_ADD()).
- Use standard expressions for date math, like: `Ship_Date + INTERVAL '3' DAY`

Output format:
{
    "schema": {
        "Column Name 1": {
            "description": "Description of column 1",
            "type": "DataType (e.g., INTEGER, VARCHAR, FLOAT)"
        },
        ...
    },
    "sql_query": "SQL query using only the schema above"
}

Important:
- Do NOT include any explanations or additional commentary—output only the dictionary.
- Ensure the output is strictly parseable as a Python dictionary with valid SQL.
"""


In [ ]:
sanity_check_prompt = """You are an expert data scientist. Your task is to check whether the following table schema and SQL statement over the schema is sufficient to answer the given question. If not, please adjust output the corrected schema and/or SQL statement using the same exact format:
Output format:
{
    "schema": {
        "Column Name 1": {
            "description": "Description of column 1",
            "type": "DataType (e.g., INTEGER, VARCHAR, FLOAT)"
        },
        ...
    },
    "sql_query": "SQL query using only the schema above"
}

Important:
- Do not include any explanations or extra text outside the dictionary.
- Your output must be directly parseable as a Python dictionary."""

In [ ]:
from processor.conductor_state import ConductorState
from processor.model.option import LLMOption
from processor.utils.string_processor import parse_code_string


def get_target_schema(
    ctx: ConductorState,
    question: str,
    schemas_to_produce=1,
    input_computation_nodes=[],
):
    ctx.logger.info(f"Getting target schema for the question {question}")
    messages = [
        {
            "role": "system",
            "content": schema_generator_system_prompt,
        },
        {"role": "user", "content": f"Question: {question}"},
    ]
    target_schemas = []
    for i in range(schemas_to_produce):
        target_schema = ctx.llm.chat(
            messages=messages,
            llm_option=LLMOption(
                seed=i,
                do_sample=True,
                temperature=0.6,
            ),
        )
        ctx.logger.info(f"=> Target schema: {target_schema}")
        try:
            target_schema = parse_code_string(target_schema)
            sanity_check_messages = [
                {
                    "role": "system",
                    "content": sanity_check_prompt,
                },
                {"role": "user", "content": f"- Question: ```{question}```\n- Schema and SQL script: ```{target_schema}```"},
            ]
            validated_target_schema = ctx.llm.chat(sanity_check_messages)
            validated_target_schema = parse_code_string(validated_target_schema)
            ctx.logger.info(f"=> Validated target schema: {validated_target_schema}")
            target_schemas.append(validated_target_schema)
        except ValueError:
            ctx.logger.error(
                "Error encountered during target schema parsing."
            )
    return ctx.computation_graph.create_node(
        computation_description="Produced target schema(s) for the given question.",
        computation_output=target_schemas,
        input_nodes=input_computation_nodes,
    )

In [15]:
target_schema_node = get_target_schema(processor.ctx, QUESTION_1)
target_schema = target_schema_node.computation_output
print(target_schema)

[2025-04-29 00:34:18] INFO in 1606367127: Getting target schema for the question Assuming all UPS ground shipments are delayed by 3 days before they are shipped (i.e., ship 3 days later than scheduled), how many items will be impacted?
[2025-04-29 00:34:59] INFO in 1606367127: => Target schema: {
  "schema": {
    "ID": {
      "description": "Unique identifier for each shipment",
      "type": "INTEGER"
    },
    "Ship_Date": {
      "description": "Scheduled shipping date",
      "type": "DATE"
    },
    "Actual_Ship_Date": {
      "description": "Actual shipping date after delay",
      "type": "DATE"
    },
    "Service_Provider": {
      "description": "Shipping service provider",
      "type": "VARCHAR"
    },
    "Quantity": {
      "description": "Number of items in the shipment",
      "type": "INTEGER"
    }
  },
  "sql_query": "SELECT COUNT(*) FROM target_table WHERE Service_Provider = 'UPS Ground Service' AND Actual_Ship_Date >= Ship_Date + INTERVAL '3' DAY"
}
[2025-04-29

In [ ]:
target_schema = {
    "ID": {"description": "Unique identifier for each shipment", "type": "INTEGER"},
    "Ship_Date": {"description": "Scheduled shipping date", "type": "DATE"},
    "Actual_Ship_Date": {
        "description": "Actual shipping date after delay",
        "type": "DATE",
    },
    "Service_Provider": {
        "description": "Shipping service provider",
        "type": "VARCHAR",
    },
    "Quantity": {
        "description": "Number of items in the shipment",
        "type": "INTEGER",
    },
}
sql_query = "SELECT SUM(Quantity) AS Impacted_Items FROM target_table WHERE Service_Provider = 'UPS Ground Service' AND Actual_Ship_Date >= Ship_Date + INTERVAL '3' DAY"

In [16]:
table_descriptions_node = processor.get_table_descriptions(DB_SCHEMA, existing_descriptions=table_descriptions)
table_descriptions = table_descriptions_node.computation_output
print(table_descriptions)

Describing tables:   0%|          | 0/5 [00:00<?, ?it/s]

[2025-04-28 05:29:11] INFO in schema_processor: Step 1: Sample rows multiple times to get different perspectives.
[2025-04-28 05:29:58] INFO in schema_processor: Step 2: Combine all perspectives.
[2025-04-28 05:30:08] INFO in schema_processor: => Overall description of table 'JI_ASN': This table contains detailed information about Advanced Shipping Notices (ASNs) at the document level, including ASN identifiers, shipment numbers, organizational IDs, shipment and delivery dates, shipment notes, and the last update timestamp.


Describing tables:  20%|██        | 1/5 [00:56<03:47, 56.83s/it]

[2025-04-28 05:30:08] INFO in schema_processor: Step 1: Sample rows multiple times to get different perspectives.
[2025-04-28 05:30:51] INFO in schema_processor: Step 2: Combine all perspectives.
[2025-04-28 05:31:04] INFO in schema_processor: => Overall description of table 'JI_ASN_CARRIER': This table provides detailed information about advanced shipping notices (ASNs) for shipments, including the carrier responsible for the shipment, the organization associated with the shipment, the domain of the shipment control ID, and the timestamp when the ASN was created or last updated.


Describing tables:  40%|████      | 2/5 [01:52<02:48, 56.01s/it]

...

[2025-04-28 05:31:48] INFO in schema_processor: Step 1: Sample rows multiple times to get different perspectives.
[2025-04-28 05:33:35] INFO in schema_processor: Step 2: Combine all perspectives.
[2025-04-28 05:34:00] INFO in schema_processor: => Overall description of table 'JI_PURCHASE_ORDER_LINE': This table provides comprehensive details on purchase order (PO) lines, including information on the organization, purchase order, line item, supplier, and financial aspects. It captures metadata such as creation and distribution timestamps, accounting dates, and statuses related to receipts, invoices, and shipments. The table also includes pricing details in different currencies and unit prices, as well as classification information for the items ordered. Additionally, it tracks the workflow status and any special conditions or actions associated with each PO line.


Describing tables:  80%|████████  | 4/5 [04:48<01:22, 82.65s/it]

[2025-04-28 05:34:00] INFO in schema_processor: Step 1: Sample rows multiple times to get different perspectives.
[2025-04-28 05:35:19] INFO in schema_processor: Step 2: Combine all perspectives.
[2025-04-28 05:35:37] INFO in schema_processor: => Overall description of table 'JI_FULFILLMENT_CENTER_TERMS_CONDITIONS': This table provides comprehensive details on purchase order terms and conditions, including acceptance instructions, payment terms, and purchasing information. It captures various aspects such as the fulfillment center key, business unit ID, organizational ID, and specific instructions related to order acceptance, shipping, and payment discounts. The table also includes information on custom payment terms and purchasing contact details.


Describing tables: 100%|██████████| 5/5 [06:25<00:00, 77.05s/it]

{'JI_ASN': 'This table contains detailed information about Advanced Shipping Notices (ASNs) at the document level, including ASN identifiers, shipment numbers, organizational IDs, shipment and delivery dates, shipment notes, and the last update timestamp.', 'JI_ASN_CARRIER': 'This table provides detailed information about advanced shipping notices (ASNs) for shipments, including the carrier responsible for the shipment, the organization associated with the shipment, the domain of the shipment control ID, and the timestamp when the ASN was created or last updated.', 'JI_ASN_LINE': 'This table provides detailed information on individual lines of an advanced shipping notice (ASN), including the quantity shipped, associated purchase order lines, organizational identifiers, and timestamps for shipment details.', 'JI_PURCHASE_ORDER_LINE': 'This table provides comprehensive details on purchase order (PO) lines, including information on the organization, purchase order, line item, supplier, an

In [ ]:
# enhanced_schemas_node = processor.get_enhanced_schemas(
#     DB_SCHEMA, table_descriptions, 3, [table_descriptions_node]
# )
# enhanced_schemas = enhanced_schemas_node.computation_output
# print(enhanced_schemas)

# Step 2: Base Table Producer

In [ ]:
target_schema = {
    "ID": {"description": "Unique identifier for each shipment", "type": "INTEGER"},
    "Ship_Date": {"description": "Scheduled shipping date", "type": "DATE"},
    "Actual_Ship_Date": {
        "description": "Actual shipping date after delay",
        "type": "DATE",
    },
    "Service_Provider": {
        "description": "Shipping service provider",
        "type": "VARCHAR",
    },
    "Quantity": {
        "description": "Number of items in the shipment",
        "type": "INTEGER",
    },
}

sql_query = "SELECT SUM(Quantity) AS Impacted_Items FROM target_table WHERE Service_Provider = 'UPS Ground Service' AND Actual_Ship_Date >= Ship_Date + INTERVAL '3' DAY"

table_descriptions = {
    "JI_ASN": "This table contains detailed information about Advanced Shipping Notices (ASNs) at the document level, including ASN identifiers, shipment numbers, organizational IDs, shipment and delivery dates, shipment notes, and the last update timestamp.",
    "JI_ASN_CARRIER": "This table provides detailed information about advanced shipping notices (ASNs) for shipments, including the carrier responsible for the shipment, the organization associated with the shipment, the domain of the shipment control ID, and the timestamp when the ASN was created or last updated.",
    "JI_ASN_LINE": "This table provides detailed information on individual lines of an advanced shipping notice (ASN), including the quantity shipped, associated purchase order lines, organizational identifiers, and timestamps for shipment details.",
    "JI_PURCHASE_ORDER_LINE": "This table provides comprehensive details on purchase order (PO) lines, including information on the organization, purchase order, line item, supplier, and financial aspects. It captures metadata such as creation and distribution timestamps, accounting dates, and statuses related to receipts, invoices, and shipments. The table also includes pricing details in different currencies and unit prices, as well as classification information for the items ordered. Additionally, it tracks the workflow status and any special conditions or actions associated with each PO line.",
    "JI_FULFILLMENT_CENTER_TERMS_CONDITIONS": "This table provides comprehensive details on purchase order terms and conditions, including acceptance instructions, payment terms, and purchasing information. It captures various aspects such as the fulfillment center key, business unit ID, organizational ID, and specific instructions related to order acceptance, shipping, and payment discounts. The table also includes information on custom payment terms and purchasing contact details.",
}

## Relevancy Check

In [ ]:
# tables_selector = """You are an experienced data scientist. You are given:
# - A table, represented by its schema, a description of what it contains, and some sample rows. The pipe character (`|`) is used as the separator for both columns and row values.
# - A target schema that needs to be constructed using one or more of the available tables.
# - A SQL statement over the target schema to answer a user's question.

# Your task is to determine whether this table is **relevant** for constructing the target schema — either fully or partially, guided by the SQL statement. A table is considered relevant if it provides **any** useful information toward fulfilling the target schema, such as:
# - Matching any of the target columns exactly,
# - Providing a column that can be transformed into a target column,
# - Contributing auxiliary information (e.g., geographic clues from `city` or `address` that help construct `Is in Bay Area`).

# Err on the side of inclusion: if you think even **one** column might help, mark the table as **relevant**.

# End your reasoning with the following exact format, to ease parsing:

# Relevant: yes/no"""

In [98]:
tables_selector = """You are an experienced data scientist. You are given:
- A table, represented by its schema, a description of what it contains, and some sample rows. The pipe character (`|`) is used as the separator for both columns and row values.
- A target schema that needs to be constructed using one or more of the available tables.
- A SQL statement over the target schema to answer a user's question.

Your task is to determine whether this table is **relevant** for constructing the target schema — even if only partially — guided by the SQL statement.

Relevance should not be judged in isolation. You are not deciding whether this table can satisfy the target schema on its own, but whether it contributes useful information toward fulfilling it when combined with other tables.

A table is considered relevant if:
- It contains columns that match, approximately match, or could be transformed into columns in the target schema.
- It holds information that supports the meaning or intent of the SQL query, even if it doesn't contain all required fields.
- It helps cover **any** part of the target schema — especially if combined with other relevant tables.

A table is not relevant **only if**:
- It has no semantically useful data that contributes to the target schema or query intent — not even partially.

Do *not* reject a table just because it lacks "critical fields" or cannot satisfy the query on its own. Your task is to identify whether it **adds value** to the final target — even a single useful column is enough.

End your reasoning with the following exact format, to ease parsing:

Relevant: yes/no"""

In [99]:
from processor.model.message import LLMMessage


def select_relevant_table_ids(
        ctx: ConductorState,
        db_schema: str,
        target_schema: list[str],
        table_descriptions: dict[str, str],
        num_rows=3,
        input_computation_nodes: list = [],
    ):
        relevant_table_ids: list[str] = []
        table_mapping = ctx.table_store.get_all_tables_in_db_schema(db_schema)

        for table_id, table in table_mapping.items():
            ctx.logger.info(f"Checking the relevance of table `{table_id}`")
            table_description = table_descriptions[table_id]
            msg: list[LLMMessage] = [
                {
                    "role": "system",
                    "content": tables_selector,
                },
                {
                    "role": "user",
                    "content": f"""- Table: ```{table.get_representation(num_rows, 42)}```
- Target schema: ```{target_schema}```
- SQL: ```{sql_query}```
- Description: ```{table_description}```""",
                },
            ]
            table_relevancy_output = ctx.llm.chat(msg)
            ctx.logger.info(f"=> Table relevancy output: {table_relevancy_output}")
            table_relevance = (
                table_relevancy_output.split("Relevant: ")[-1].lower().strip()
            )
            if table_relevance.startswith("yes"):
                ctx.logger.info(f"==> Conclusion: The table is considered relevant!")
                relevant_table_ids.append(table_id)
        return ctx.computation_graph.create_node(
            computation_description="Selected relevant table IDs for the given target schema.",
            computation_output=relevant_table_ids,
            input_nodes=input_computation_nodes,
        )

In [100]:
relevant_table_ids_node = select_relevant_table_ids(
    processor.ctx,
    DB_SCHEMA,
    target_schema,
    table_descriptions,
    # 3,
    # [target_schema_node, table_descriptions_node],
)
relevant_table_ids = relevant_table_ids_node.computation_output
print(relevant_table_ids)

[2025-04-29 02:56:29] INFO in 2519414222: Checking the relevance of table `JI_ASN`
[2025-04-29 02:57:09] INFO in 2519414222: => Table relevancy output: The provided table contains shipment-related information such as `SHIPMENT_DATE` and `DELIVERY_DATE`, which are relevant to the target schema's `Ship_Date` and `Actual_Ship_Date`. However, the table does not contain the `Service_Provider`, `Quantity`, or any direct identifier that matches the `ID` in the target schema. 

While the table does not have all the necessary fields, it still provides valuable information related to the shipment dates, which can be used in combination with other tables to construct the target schema. Specifically, the `SHIPMENT_DATE` and `DELIVERY_DATE` can be used to derive the `Ship_Date` and `Actual_Ship_Date`.

Given that the table contributes useful information towards fulfilling parts of the target schema, it is relevant.

Relevant: yes
[2025-04-29 02:57:09] INFO in 2519414222: ==> Conclusion: The table i

## UNION

In [109]:
BEFORE_UNION_DB_SCHEMA = "BEFORE_UNION_DB_SCHEMA"

### OPERATIONS

In [113]:
from tqdm import tqdm
relevant_table_ids = ['JI_ASN', 'JI_ASN_CARRIER']
try:
    processor.ctx.table_store.create_db_schema(BEFORE_UNION_DB_SCHEMA)
except:
    pass
for table_name in tqdm(relevant_table_ids):
    try:
        table = processor.ctx.table_store.get_table(DB_SCHEMA, table_name)
        processor.ctx.table_store.add_table(BEFORE_UNION_DB_SCHEMA, table_name, table)
    except:
        pass
processor.ctx.table_store.checkpoint()

100%|██████████| 2/2 [00:00<00:00, 22671.91it/s]


In [114]:
row_extender_step_1 = """You are an experienced data scientist. You are given:
- A list of tables, each with its schema, a short description, and a few sample rows.
- The pipe character (`|`) is used to separate both column names and values.

Your task is to **analyze and describe** what each table represents, and then identify **which tables describe the same kind of real-world entity or object** (such as people, products, companies, events, etc.).

Only group tables that:
- Refer to the same kind of entity
- Can be combined via **row extension** (i.e., vertical stacking)
- Even if the columns are not exactly the same, their rows should be logically stackable (e.g., two tables of products with different attributes)

Do **not** group tables that refer to different concepts/entities, even if they share similar-looking columns.

Finish with a list of compatible groups like:
Row extension groups: Group 1: Table_0, Table_2 Group 2: Table_3, Table_4 ... (or none if no combinations are found)"""
row_extender_step_2 = """You are an experienced data scientist. You have already analyzed the tables and identified which ones can be unioned together because they refer to the same kind of real-world entity.

You are given:
- A list of tables (description + schemas + samples)
- Your own prior reasoning and a list of union groups (e.g., Group 1: Table_0, Table_2)

Your job is to create a JSON plan that shows how each **group** can be unioned.

Instructions:
- ONLY process tables that belong to the identified union groups based on your final conclusion.
- IGNORE any tables that are not part of any group.
- IGNORE any groups that only consist of a single table.
- For each group:
    - Create a **unified schema** by merging **semantically equivalent** columns across the group's tables (e.g., "Customer_Rating" and "RATING" → "Rating")
    - Use **simple, general, and meaningful** names for the unified columns (e.g., "Phone", "Address", "Rating", "Reviews")
    - For each table in the group, create a mapping from its original column names to the unified schema
    - It's OK if some original columns don't map — just omit them
    - Ensure there are no duplicate concepts in the unified schema

Output directly the following format without extra texts or explanations:

Format if row extension groups exist:
```json
[
    {
        "Output Table ID": "Union_1",
        "Tables": ["Table_0", "Table_2"],
        "Unified Schema": ["Column1", "Column2", ...],
        "Mappings": {
            "Table_0": {"OrigColA": "Column1", "OrigColB": "Column2", ...},
            "Table_2": {"ColX": "Column1", "ColY": "Column2", ...}
        }
    }
]```

Format if row extension groups are empty/none:
```json
[]```"""

In [115]:
def format_available_tables(
        ctx,
        db_schema: str,
        num_rows: int,
        table_descriptions: dict[str, str],
    ):
        available_tables_formatted = ""
        table_mappings = ctx.table_store.get_all_tables_in_db_schema(db_schema)
        for table_id, table in table_mappings.items():
            table_description = table_descriptions[table_id]
            available_tables_formatted += f"""- {table_id} ({table_description}):
```{table.get_representation(num_rows, 42)}```\n"""

        available_tables_formatted = available_tables_formatted.strip()
        ctx.logger.info(f"=> available_tables_formatted: {available_tables_formatted}")
        return available_tables_formatted

In [116]:
from processor.utils.string_processor import parse_code_string
def produce_union_operations(
        ctx,
        db_schema: str,
        table_descriptions: dict[str, str],
        num_rows=3,
        input_computation_nodes: list = [],
    ):
        """
        Returns a list of operations to union tables (if any) within the DB schema.

        Args:
            ctx (ConductorState): Conductor state object.
            db_schema (str): The DB schema to get the tables from.
            table_descriptions (dict[str,str]): The descriptions of the tables.
            num_rows (int): Number of rows to sample from each table.
            input_computation_nodes (Node): A list of input nodes to keep track of computation.
        Returns:
            Output (Node[Any]): Computation node consisting of the union operations.
        """
        available_tables_formatted = format_available_tables(
            ctx,
            db_schema,
            num_rows,
            table_descriptions,
        )

        msg = [
            {
                "role": "system",
                "content": row_extender_step_1,
            },
            {"role": "user", "content": available_tables_formatted},
        ]
        reasoning = ctx.llm.chat(msg)
        ctx.logger.info(f"=> reasoning: {reasoning}")
        msg = [
            {
                "role": "system",
                "content": row_extender_step_2,
            },
            {
                "role": "user",
                "content": f"- Tables: {available_tables_formatted}\n\n- Reasoning: {reasoning}",
            },
        ]
        llm_output = ctx.llm.chat(msg)
        operations = parse_code_string(llm_output)
        ctx.logger.info(f"=> operations: {operations}")
        return ctx.computation_graph.create_node(
            computation_description="Produced union operations.",
            computation_output=operations,
            input_nodes=input_computation_nodes,
        )

In [117]:
union_operations_node = produce_union_operations(
    processor.ctx,
    BEFORE_UNION_DB_SCHEMA,
    table_descriptions,
    3,
    # [target_schema_node, table_descriptions_node],
)
union_operations = union_operations_node.computation_output
print(union_operations)

[2025-04-29 03:04:50] INFO in 2840427047: => available_tables_formatted: - JI_ASN (This table contains detailed information about Advanced Shipping Notices (ASNs) at the document level, including ASN identifiers, shipment numbers, organizational IDs, shipment and delivery dates, shipment notes, and the last update timestamp.):
```col: ASN_ID | ASN_SHIPMENT_NUMBER | ORG_ID | SHIPMENT_DATE | DELIVERY_DATE | SHIPMENT_NOTES | ELT_TS
sample row 1: 11473177 | CHIL79782 | 1962414 | 2022-05-31 04:00:00 | 2022-05-31 04:00:00 | None | 2023-11-04 03:30:32.330000
sample row 2: 9820261 | CHIL75600 | 1962414 | 2021-10-28 04:00:00 | 2021-10-28 04:00:00 | None | 2023-11-04 03:30:32.330000
sample row 3: 13227851 | CHIL83297 | 1962414 | 2022-12-14 05:00:00 | 2022-12-22 05:00:00 | None | 2023-11-04 03:30:32.330000```
- JI_ASN_CARRIER (This table provides detailed information about advanced shipping notices (ASNs) for shipments, including the carrier responsible for the shipment, the organization associat

### Execution

In [118]:
union_operations = [
    {
        "Output Table ID": "Union_1",
        "Tables": ["JI_ASN", "JI_ASN_CARRIER"],
        "Unified Schema": [
            "ASN_ID",
            "ASN_SHIPMENT_NUMBER",
            "ORG_ID",
            "SHIPMENT_DATE",
            "DELIVERY_DATE",
            "SHIPMENT_NOTES",
            "CARRIER",
            "SHIPMENT_CONTROL_ID",
            "ELT_TS",
        ],
        "Mappings": {
            "JI_ASN": {
                "ASN_ID": "ASN_ID",
                "ASN_SHIPMENT_NUMBER": "ASN_SHIPMENT_NUMBER",
                "ORG_ID": "ORG_ID",
                "SHIPMENT_DATE": "SHIPMENT_DATE",
                "DELIVERY_DATE": "DELIVERY_DATE",
                "SHIPMENT_NOTES": "SHIPMENT_NOTES",
                "ELT_TS": "ELT_TS",
            },
            "JI_ASN_CARRIER": {
                "ASN_ID": "ASN_ID",
                "ORG_ID": "ORG_ID",
                "CARRIER": "CARRIER",
                "SHIPMENT_CONTROL_ID": "SHIPMENT_CONTROL_ID",
                "ELT_TS": "ELT_TS",
            },
        },
    }
]

In [119]:
unioned_tables_node = processor.run_union_operations(
    processor.ctx.table_store.get_all_tables_in_db_schema(BEFORE_UNION_DB_SCHEMA),
    union_operations,
    # [union_operations_node],
)
unioned_tables = unioned_tables_node.computation_output
print(unioned_tables)

{'Union_1': <processor.table.representation.impl.df_table.DFTable object at 0x7f2adb23f530>}


In [120]:
import pandas as pd
import numpy as np

def clean_duplicate_ids(df: pd.DataFrame, id_col='ID') -> pd.DataFrame:
    df = df.copy()
    # 1) Record original dtypes
    orig_dtypes = df.dtypes.copy()

    non_id_cols = [col for col in df.columns if col != id_col]
    # 2) Ensure df itself uses pandas’ nullable types
    for col in non_id_cols:
        dt = orig_dtypes[col]
        if dt.kind in ('i',):        # integer
            df[col] = df[col].astype("Int64")
        elif dt.kind in ('f',):      # float
            df[col] = df[col].astype("Float64")
        elif dt == object:           # string/object
            df[col] = df[col].astype("string")

    grouped = df.groupby(id_col, dropna=False)
    cleaned_rows = []

    for id_val, group in grouped:
        if len(group) == 1:
            cleaned_rows.append(group.iloc[0].to_dict())
            continue

        values_dict = {}
        misalign_count = total_checks = 0

        for col in non_id_cols:
            non_null_vals = group[col].dropna().unique()
            values_dict[col] = non_null_vals.tolist()
            if len(non_null_vals) > 1:
                misalign_count += 1
            if len(non_null_vals) >= 1:
                total_checks += 1

        if total_checks > 0 and misalign_count == total_checks:
            cleaned_rows.extend(group.to_dict(orient="records"))
            continue

        max_len = max(len(v) for v in values_dict.values())
        if max_len <= 1:
            combined = {col: (vals[0] if vals else np.nan)
                        for col, vals in values_dict.items()}
            combined[id_col] = id_val
            cleaned_rows.append(combined)
        else:
            for i in range(max_len):
                row = {}
                for col, vals in values_dict.items():
                    if len(vals) == 1:
                        row[col] = vals[0]
                    elif i < len(vals):
                        row[col] = vals[i]
                    else:
                        row[col] = np.nan
                row[id_col] = id_val
                cleaned_rows.append(row)

    # 3) Build the result and restore nullable dtypes
    result = pd.DataFrame(cleaned_rows)[df.columns].reset_index(drop=True)
    for col in result.columns:
        orig_dt = orig_dtypes[col]
        if orig_dt.kind in ('i',):
            result[col] = result[col].astype("Int64")
        elif orig_dt.kind in ('f',):
            result[col] = result[col].astype("Float64")
        elif orig_dt == object:
            result[col] = result[col].astype("string")

    return result


In [121]:
for table_name in unioned_tables:
    if table_name.startswith('Union'):
        unioned_tables[table_name].data = clean_duplicate_ids(unioned_tables[table_name].data, unioned_tables[table_name].data.columns[0])

In [122]:
y = unioned_tables['Union_1'].data
# y = y[y['ASN_ID'] == 17920751]
y

,ASN_ID,ASN_SHIPMENT_NUMBER,ORG_ID,SHIPMENT_DATE,DELIVERY_DATE,SHIPMENT_NOTES,CARRIER,SHIPMENT_CONTROL_ID,ELT_TS
0,7133325,CHIL67664,1962414,2020-08-11 04:00:00,2020-08-11 04:00:00,<NA>,<NA>,<NA>,2023-11-04 03:30:32.330
1,7201307,CHIL67884,1962414,2020-08-20 04:00:00,2020-08-25 04:00:00,<NA>,<NA>,<NA>,2023-11-04 03:30:32.330
2,7201308,CHIL67881,1962414,2020-08-20 04:00:00,2020-08-25 04:00:00,<NA>,<NA>,<NA>,2023-11-04 03:30:32.330
3,7201368,CHIL67913,1962414,2020-08-24 04:00:00,2020-08-25 04:00:00,<NA>,<NA>,<NA>,2023-11-04 03:30:32.330
4,7201542,CHIL67879,1962414,2020-08-20 04:00:00,2020-08-25 04:00:00,<NA>,<NA>,<NA>,2023-11-04 03:30:32.330
...,...,...,...,...,...,...,...,...,...
1057,19360427,CHIL92492,1962414,2024-06-07 04:00:00,2024-06-25 04:00:00,<NA>,Upsg - Ups Ground Service,8920852,2024-06-25 06:00:31.921
1058,19377664,CHIL92722,1962414,2024-06-24 04:00:00,<NA>,<NA>,FAST,8928612,2024-06-27 01:45:25.885
1059,19557965,CHIL92948,1962414,2024-07-11 04:00:00,2024-07-11 04:00:00,<NA>,Dafg - Dayton Freight Lines,9019853,2024-07-12 08:45:27.254
1060,19573864,CHIL92959,1962414,2024-07-12 04:00:00,<NA>,<NA>,FAST,9029611,2024-07-13 03:00:26.207


In [124]:
# Save to DB
try:
    processor.ctx.table_store.create_db_schema(DB_SCHEMA_AFTER_UNION)
except:
    pass
for table_id, table in unioned_tables.items():
    processor.ctx.table_store.add_table(DB_SCHEMA_AFTER_UNION, table_id, table, True, False)
processor.ctx.table_store.checkpoint()

In [125]:
join_desc_sys_prompt = """You are given a table that was formed using multiple tables. Each of these source tables have its own description.

Your goal is to describe briefly what this table represents (roughly as long as the individual source table descriptions)."""

In [126]:
for union_op in union_operations:
    tables = union_op["Tables"]
    union_table = union_op["Output Table ID"]
    concatenated_descriptions = ""
    for table in tables:
        table_desc = table_descriptions[table]
        concatenated_descriptions += f"- {table_desc}\n"
    concatenated_descriptions = concatenated_descriptions.strip()
    msg = [
        {'role': 'system', 'content': join_desc_sys_prompt},
        {'role': 'user', 'content': f"- Table: ```{processor.ctx.table_store.get_table(DB_SCHEMA_AFTER_UNION, union_table).get_representation(3)}```\n- Individual descriptions: ```{concatenated_descriptions}```"},
    ]
    llm_output = processor.ctx.llm.chat(msg)
    table_descriptions[union_table] = llm_output

## JOIN

In [ ]:
table_descriptions = {
    "JI_ASN": "This table contains detailed information about Advanced Shipping Notices (ASNs) at the document level, including ASN identifiers, shipment numbers, organizational IDs, shipment and delivery dates, shipment notes, and the last update timestamp.",
    "JI_ASN_CARRIER": "This table provides detailed information about advanced shipping notices (ASNs) for shipments, including the carrier responsible for the shipment, the organization associated with the shipment, the domain of the shipment control ID, and the timestamp when the ASN was created or last updated.",
    "JI_ASN_LINE": "This table provides detailed information on individual lines of an advanced shipping notice (ASN), including the quantity shipped, associated purchase order lines, organizational identifiers, and timestamps for shipment details.",
    "JI_PURCHASE_ORDER_LINE": "This table provides comprehensive details on purchase order (PO) lines, including information on the organization, purchase order, line item, supplier, and financial aspects. It captures metadata such as creation and distribution timestamps, accounting dates, and statuses related to receipts, invoices, and shipments. The table also includes pricing details in different currencies and unit prices, as well as classification information for the items ordered. Additionally, it tracks the workflow status and any special conditions or actions associated with each PO line.",
    "JI_FULFILLMENT_CENTER_TERMS_CONDITIONS": "This table provides comprehensive details on purchase order terms and conditions, including acceptance instructions, payment terms, and purchasing information. It captures various aspects such as the fulfillment center key, business unit ID, organizational ID, and specific instructions related to order acceptance, shipping, and payment discounts. The table also includes information on custom payment terms and purchasing contact details.",
    "Union_1": "This table consolidates detailed information about Advanced Shipping Notices (ASNs) at both the document and shipment levels. It includes fields such as ASN identifier (`ASN_ID`), organizational ID (`ORG_ID`), shipment date (`SHIPMENT_DATE`), delivery date (`DELIVERY_DATE`), shipment notes (`SHIPMENT_NOTES`), the timestamp of the last update (`ELT_TS`), the carrier responsible for the shipment (`CARRIER`), the domain associated with the shipment control ID (`DOMAIN`), and the shipment control ID itself (`SHIPMENT_CONTROL_ID`). The data spans various shipment events and captures key details like dates, notes, and carrier information, providing a comprehensive view of the shipment process.",
}

In [ ]:
base_table_producer_prompts = {
        "tables_selector": """You are an experienced data scientist. You are given:
- A table, represented by its schema, a description of what it contains, and some sample rows. The pipe character (`|`) is used as the separator for both columns and row values.
- A target schema that needs to be constructed using one or more of the available tables.

Your task is to determine whether this table is **relevant** for constructing the target schema — either fully or partially. A table is considered relevant if it provides **any** useful information toward fulfilling the target schema, such as:
- Matching any of the target columns exactly,
- Providing a column that can be transformed into a target column,
- Contributing auxiliary information (e.g., geographic clues from `city` or `address` that help construct `Is in Bay Area`).

Err on the side of inclusion: if you think even **one** column might help, mark the table as **relevant**.

End your reasoning with the following exact format, to ease parsing:

Relevant: yes/no""",
        "row_extender_step_1": """You are an experienced data scientist. You are given:
- A list of tables, each with its schema, a short description, and a few sample rows.
- The pipe character (`|`) is used to separate both column names and values.

Your task is to **analyze and describe** what each table represents, and then identify **which tables describe the same kind of real-world entity or object** (such as people, products, companies, events, etc.).

Only group tables that:
- Refer to the same kind of entity
- Can be combined via **row extension** (i.e., vertical stacking)
- Even if the columns are not exactly the same, their rows should be logically stackable (e.g., two tables of products with different attributes)

Do **not** group tables that refer to different concepts/entities, even if they share similar-looking columns.

Finish with a list of compatible groups like:
Row extension groups: Group 1: Table_0, Table_2 Group 2: Table_3, Table_4 ... (or none if no combinations are found)""",
        "row_extender_step_2": """You are an experienced data scientist. You have already analyzed the tables and identified which ones can be unioned together because they refer to the same kind of real-world entity.

You are given:
- A list of tables (description + schemas + samples)
- Your own prior reasoning and a list of union groups (e.g., Group 1: Table_0, Table_2)

Your job is to create a JSON plan that shows how each group can be unioned.

Instructions:
- For each group, create a **unified schema** by merging **semantically equivalent** columns (e.g., "Customer_Rating" and "RATING" should both become "Rating")
- Use **simple, general, and meaningful** names for the unified columns (e.g., "Phone", "Address", "Rating", "Reviews")
- For each table, create a mapping from its original column names to the unified schema
- It's okay if some original columns do not exist in the unified schema — just leave them unmapped
- Do not include duplicate columns in the unified schema — each concept should appear only once

Output directly the following format without extra texts or explanations:

Format if row extension groups exist:
```json
[
    {
        "Output Table ID": "Union_1",
        "Tables": ["Table_0", "Table_2"],
        "Unified Schema": ["Column1", "Column2", ...],
        "Mappings": {
            "Table_0": {"OrigColA": "Column1", "OrigColB": "Column2", ...},
            "Table_2": {"ColX": "Column1", "ColY": "Column2", ...}
        }
    }
]```

Format if row extension groups are empty/none:
```json
[]```""",
        "join_planner": """You are a highly skilled data engineer. You are given:
- A list of tables (with descriptions, schemas, and sample rows)
- The goal is to **join all tables** together into a final unified table by **step-wise horizontal merging**.

Assumptions:
- All tables should be joinable via appropriate key columns, either directly or through intermediate tables.
- You can choose any join order as long as all tables are included by the end.
- You should identify the most appropriate **key columns** for joining each pair of tables based on semantics or value similarity.
- The operations will be carried out using either SQL or semantic joins.

Your task:
- Construct a step-by-step join plan as a **list of operations**, where each operation joins two tables (or previous join results).
- Each step should specify:
    - The two input tables, one of which may be a join result from the prior step.
    - The columns being used for the join
    - The resulting table name for that step (e.g., "Join_1", "Join_2", etc.)

Output your answer directly as a JSON object with the following format without any extra explanations or formatting:

```json
[
    {
        "Join Result": "Join_1",
        "Left Table": "Table_A",
        "Right Table": "Table_B",
        "Left Join Key": "Column_X",
        "Right Join Key": "Column_Y"
    },
    {
        "Join Result": "Join_2",
        "Left Table": "Join_1",
        "Right Table": "Table_C",
        "Left Join Key": "UserID",
        "Right Join Key": "Customer_ID"
    }
]```

Remember, no comments, extra explanations, or formatting.""",
        "classification_prompt": """You are a highly skilled data engineer. You are given:
- A description of a join operation between two tables.
- Sample values for each join key column from both tables.

Your task is to classify whether the join can be performed using a standard SQL join (e.g., matching IDs or exactly matching names), or if it requires a *semantic join*. A semantic join is needed when the values differ in representation — for example, abbreviations, name variations, different formats, or different languages — and require normalization, transformation, or external knowledge to align correctly. Please note that null value handling does not consitute semantic joins.

Carefully examine the values. If they seem to not be exactly equal (not because of the fact that they are sample values), and some interpretation or resolution is needed to make the join work, it is a semantic join.

At the end of your reasoning, respond in the following format (for easy parsing):

- Operation classification: standard or semantic""",
        "std_join": """You are a highly skilled data engineer.
You are given two tables, represented by their IDs, descriptions, schemas, and sample rows.

Your goal is to create an ANSI SQL script to inner join these tables through a given left and right join keys. Refer to the IDs as identifiers in the script.

Output the SQL script directly without any extra formatting or explanation.""",
}

### 1. Produce Join Operations

In [ ]:
def __format_available_tables(
        ctx,
        db_schema: str,
        num_rows: int,
        table_descriptions: dict[str, str],
    ):
        available_tables_formatted = ""
        table_mappings = ctx.table_store.get_all_tables_in_db_schema(db_schema)
        for table_id, table in table_mappings.items():
            table_description = table_descriptions[table_id]
            available_tables_formatted += f"""- {table_id} ({table_description}):
```{table.get_representation(num_rows, 42)}```\n"""

        available_tables_formatted = available_tables_formatted.strip()
        ctx.logger.info(f"=> available_tables_formatted: {available_tables_formatted}")
        return available_tables_formatted

In [ ]:
join_planner = """You are a highly skilled data engineer. You are given:
- A list of tables (with descriptions, schemas, and sample rows)
- The goal is to **join all tables** together into a final unified table in a **step-wise horizontal merging**.

Assumptions:
- All tables should be joinable via appropriate key columns, either directly or through intermediate tables.
- You can choose any join order as long as all tables are included by the end.
- You should identify the most appropriate **key columns** for joining each pair of tables based on semantics or value similarity.
- The operations will be carried out using either SQL or semantic joins.

Your task:
- Construct a step-by-step join plan as a **list of operations**, where each operation joins two tables (or previous join results).
- Each step should specify:
    - The two input tables, one of which may be a join result from the prior step.
    - The columns being used for the join
    - The resulting table name for that step (e.g., "Join_1", "Join_2", etc.)

Output your answer directly as a JSON object with the following format without any extra explanations or formatting:

```json
[
    {
        "Join Result": "Join_1",
        "Left Table": "Table_A",
        "Right Table": "Table_B",
        "Left Join Key": "Column_X",
        "Right Join Key": "Column_Y"
    },
    {
        "Join Result": "Join_2",
        "Left Table": "Join_1",
        "Right Table": "Table_C",
        "Left Join Key": "UserID",
        "Right Join Key": "Customer_ID"
    }
]```

Remember, no comments, extra explanations, or formatting."""

In [ ]:
from processor.utils.string_processor import parse_code_string


def produce_join_operations(
        ctx,
        db_schema: str,
        table_descriptions: dict[str, str],
        num_rows=3,
        input_computation_nodes: list = [],
    ):
        available_tables_formatted = __format_available_tables(
            ctx, db_schema, num_rows, table_descriptions
        )
        msg = [
            {"role": "system", "content": base_table_producer_prompts["join_planner"]},
            {"role": "user", "content": available_tables_formatted},
        ]
        join_operations: list[dict[str, str]] = parse_code_string(ctx.llm.chat(msg))
        ctx.logger.info(f"Join operations: {join_operations}")
        return ctx.computation_graph.create_node(
            computation_description="Produced join operations",
            computation_output=join_operations,
            input_nodes=input_computation_nodes,
        )

In [76]:
y = processor.ctx.table_store.get_all_tables_in_db_schema(DB_SCHEMA_AFTER_UNION)
purchase = y['JI_PURCHASE_ORDER_LINE'].data
union1 = y['Union_1'].data

In [78]:
union1.head(2)

,ASN_ID,ORG_ID,SHIPMENT_DATE,DELIVERY_DATE,SHIPMENT_NOTES,ELT_TS,CARRIER,DOMAIN,SHIPMENT_CONTROL_ID
0,7133325,1962414,2020-08-11 04:00:00,2020-08-11 04:00:00,<NA>,2023-11-04 03:30:32.330,<NA>,<NA>,<NA>
1,7201307,1962414,2020-08-20 04:00:00,2020-08-25 04:00:00,<NA>,2023-11-04 03:30:32.330,<NA>,<NA>,<NA>


In [79]:
purchase.head(2)

,ORG_ID,PO_ID,PO_LINE_ID,DEPT_KEY,SUPPLIER_KEY,ITEM_KEY,PO_NUMBER,CONTRACT_ID,CONTRACT_NUMBER,QUANTITY,...,GRAND_TOTAL_DOCUMENT_AMOUNT,GRAND_TOTAL_DOCUMENT_CURRENCY,GRAND_TOTAL_DOCUMENT_EXCHANGE_RATE,GRAND_TOTAL_USD,ELT_TS,FULFILLMENT_CENTER_KEY,CATEGORY_LEVEL_1_NAME,CATEGORY_LEVEL_1_UNSPSC,CATEGORY_LEVEL_2_NAME,CATEGORY_LEVEL_2_UNSPSC
0,1962414,36249932,123528743,444891,576161,30359607,W940090,-1,None,1.0,...,564.25,USD,1.0,564.25,2025-01-18 07:15:17.299,6094791,Office Equipment and Accessories and Supplies,44000000,Office machines and their supplies and accesso...,44100000
1,1962414,65594295,220674571,443589,576161,28230762,G264397,-1,None,1.0,...,17.24,USD,1.0,17.24,2025-01-18 07:15:17.299,6094791,Domestic Appliances and Supplies and Consumer ...,52000000,Domestic kitchenware and kitchen supplies,52150000


In [ ]:
join_operations_node = produce_join_operations(
    processor.ctx,
    DB_SCHEMA_AFTER_UNION,
    table_descriptions,
    3,
    # [union_operations_node],
)
join_operations = join_operations_node.computation_output
print(join_operations)

[2025-04-29 01:20:50] INFO in 698260742: => available_tables_formatted: - JI_PURCHASE_ORDER_LINE (This table provides comprehensive details on purchase order (PO) lines, including information on the organization, purchase order, line item, supplier, and financial aspects. It captures metadata such as creation and distribution timestamps, accounting dates, and statuses related to receipts, invoices, and shipments. The table also includes pricing details in different currencies and unit prices, as well as classification information for the items ordered. Additionally, it tracks the workflow status and any special conditions or actions associated with each PO line.):
```col: ORG_ID | PO_ID | PO_LINE_ID | DEPT_KEY | SUPPLIER_KEY | ITEM_KEY | PO_NUMBER | CONTRACT_ID | CONTRACT_NUMBER | QUANTITY | EXTENDED_PRICE | CREATED_TS | DISTRIBUTION_TS | EXPORT_TS | LASTREVISION_TS | ORIGINALREVISION_TS | WORKFLOWCOMPLETED_TS | ACCOUNTING_DATE | USER_OWNER_KEY | USER_SUBMITTER_KEY | EXTERNAL_PO_ID | L

### 2. Execute Join

In [ ]:
join_operations = [
    {
        "Join Result": "Join_1",
        "Left Table": "JI_PURCHASE_ORDER_LINE",
        "Right Table": "Union_1",
        "Left Join Key": "ORG_ID",
        "Right Join Key": "ORG_ID",
    }
]

In [54]:
std_join = """You are a highly skilled data engineer.

You are given two tables, each described by an ID, a textual description, a schema, and sample rows.

Your task is to write an ANSI SQL script that performs an INNER JOIN between the two tables using the specified left and right join keys. 

- Use the table IDs as identifiers in the SQL.
- Select all columns from both tables.
- Output only the SQL script, without any extra text, formatting, or explanation."""

In [ ]:
from processor.utils.string_processor import parse_sql_string


def run_std_join_operation(
    ctx,
    left_table_id: str,
    right_table_id: str,
    left_table,
    right_table,
    left_join_key: str,
    right_join_key: str,
    input_nodes = [],
):
    """Runs a single standard join operation."""
    msg = [
        {
            "role": "system",
            "content": std_join,
        },
        {
            "role": "user",
            "content": f"""- Left table (ID: {left_table_id}; join key: {left_join_key}): {left_table.get_representation(3, 42)}
- Right table (ID: {right_table_id}; join key: {right_join_key}): {right_table.get_representation(3, 42)}""",
        },
    ]
    sql_script = parse_sql_string(ctx.llm.chat(msg))

    ctx.logger.info(f"=> Executing SQL: {sql_script}")
    joined_table = ctx.table_store.execute_sql_query(
        sql_query=sql_script,
        tables_involved={
            left_table_id: left_table,
            right_table_id: right_table,
        },
    )
    return ctx.computation_graph.create_node(
        f"Executing this SQL script for a standard join operation:\n{sql_script}",
        joined_table,
        input_nodes,
    )

In [ ]:
def run_join_operations(
    ctx,
    table_mapping: dict,
    operations: list[dict[str, str]],
    num_values=3,
    input_computation_nodes = [],
):
    """
    Returns a list of operations to join tables (if any) within the DB schema.

    Args:
        ctx (ConductorState): Conductor state object.
        table_mappings (dict[str,AbstractTable]): The mapping between table IDs and table objects.
        operations (list[dict[str, Any]]): The join operations.
        num_values (int): Number of rows to sample for each table.
        input_computation_nodes (Node): A list of input nodes to keep track of computation.
    Returns:
        Output (Node[dict[str, AbstractTable]]): Computation node consisting of the table mappings after the operations have been applied.
    """

    # Ensure non-mutability of the original object
    table_mapping_copy = {k: v.copy() for k, v in table_mapping.items()}
    extra_input_nodes: list = []
    for operation in operations:
        join_table_id: str = operation["Join Result"]
        left_table_id: str = operation["Left Table"]
        right_table_id: str = operation["Right Table"]
        left_join_key: str = operation["Left Join Key"]
        right_join_key: str = operation["Right Join Key"]

        left_key_samples = table_mapping_copy[left_table_id].get_attribute_values(
            attr_name=left_join_key,
            num_values=num_values,
            random_seed=42,
        )
        right_key_samples = table_mapping_copy[right_table_id].get_attribute_values(
            attr_name=right_join_key,
            num_values=num_values,
            random_seed=42,
        )

        msg = [
            {
                "role": "system",
                "content": base_table_producer_prompts["classification_prompt"],
            },
            {
                "role": "user",
                "content": f"""- Samples of left join key ({left_join_key}): {left_key_samples}
- Samples of right join key ({right_join_key}): {right_key_samples}""",
            },
        ]
        classification_result = ctx.llm.chat(msg)
        ctx.logger.info(f"=> classification_result: {classification_result}")
        join_node = run_std_join_operation(
            ctx,
            left_table_id,
            right_table_id,
            table_mapping_copy[left_table_id],
            table_mapping_copy[right_table_id],
            left_join_key,
            right_join_key,
            input_computation_nodes,
        )
        extra_input_nodes.append(join_node)
        joined_table = join_node.computation_output
        table_mapping_copy[join_table_id] = joined_table
        del table_mapping_copy[left_table_id]
        del table_mapping_copy[right_table_id]
    return ctx.computation_graph.create_node(
        "Ran join operations over the tabless",
        table_mapping_copy,
        input_computation_nodes + extra_input_nodes,
    )

In [ ]:
join_results_node = run_join_operations(
    processor.ctx,
    processor.ctx.table_store.get_all_tables_in_db_schema(DB_SCHEMA_AFTER_UNION),
    join_operations,
    5,
    # [join_operations_node],
)
join_results = join_results_node.computation_output
print(join_results)

[2025-04-29 01:31:05] INFO in 2211558722: => classification_result: - Operation classification: standard
[2025-04-29 01:31:47] INFO in 2465876245: => Executing SQL: 
SELECT 
    JI_PURCHASE_ORDER_LINE.*, 
    Union_1.ASN_ID, 
    Union_1.SHIPMENT_DATE, 
    Union_1.DELIVERY_DATE, 
    Union_1.SHIPMENT_NOTES, 
    Union_1.ELT_TS, 
    Union_1.CARRIER, 
    Union_1.DOMAIN, 
    Union_1.SHIPMENT_CONTROL_ID
FROM 
    JI_PURCHASE_ORDER_LINE
INNER JOIN 
    Union_1
ON 
    JI_PURCHASE_ORDER_LINE.ORG_ID = Union_1.ORG_ID;



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

{'Join_1': <processor.table.representation.impl.df_table.DFTable object at 0x7f2adb23ffe0>}


# Combine

In [128]:
# base_table = join_results['Join_1']
base_table = processor.ctx.table_store.get_table(DB_SCHEMA_AFTER_UNION, 'Union_1')

In [129]:
DB_SCHEMA_FINAL = "db_schema_final"
processor.ctx.table_store.create_db_schema(DB_SCHEMA_FINAL)
processor.ctx.table_store.add_table(
    DB_SCHEMA_FINAL,
    "base_table",
    base_table,
    True,
    True,
)

# Base Table Reducer

In [6]:
target_schema = {
    "ID": {"description": "Unique identifier for each shipment", "type": "INTEGER"},
    "Ship_Date": {"description": "Scheduled shipping date", "type": "DATE"},
    "Actual_Ship_Date": {
        "description": "Actual shipping date after delay",
        "type": "DATE",
    },
    "Service_Provider": {"description": "Shipping service provider", "type": "VARCHAR"},
    "Quantity": {"description": "Number of items in the shipment", "type": "INTEGER"},
}
sql_query = "SELECT SUM(Quantity) AS Impacted_Items FROM target_table WHERE Service_Provider = 'UPS Ground Service' AND Actual_Ship_Date >= Ship_Date + INTERVAL '3' DAY"

In [7]:
base_table_reducer_prompts = {
    "python_column_extractor": """You are an experienced data scientist. Given a table represented as a Pandas DataFrame, your task is to write a Python function named generate_column with no arguments except for the dataframe itself that returns a list representing the values of the new column based on this table. You are also given a question user has, which will help you determine what kind of computations that need to be done (e.g., adding time delta to the row values). Each element of the list should correspond to a row in the DataFrame.

Constraints:
- Only use columns that are present in the input DataFrame.
- To be safe, you should convert data types of the columns in your code before performing computation.
- Handle missing (null) values carefully and appropriately.
- Use regex to determine equality instead of "==".

Output the function directly without any extra explanations or formattings.""",
    "column_projection": """You are a helpful data scientist.

You will be provided with:
- A source table called SRC that is represented by its schema and some sample rows.
- A target schema that we will transform the schema of source table into it in a step-by-step manner.
- A question that we want to answer, which was used to form the target schema.
- A column from the target schema as the current target column.

Your goal is to determine whether to select a certain column from SRC or extract information from certain column(s) from SRC to form the target column (even as simple as adding time delta to each row).
Extract_column can relies on external tools such as Python code interpreter, SQL processor, or LLM.

When using extract_column, always include all columns from SRC that are required to perform the extraction, even if their role seems minor or indirect. These can include helper columns (e.g., timestamps for computing durations, ZIP codes for locations, etc.).

The output format for selecting a certain column:
{
    "operation": "select_column",
    "description": "Select SRC.Restaurant ID."
    "columns_involved": ["Restaurant ID"],
}

While for extracting information from certain column(s):
{
    "operation": "extract_column",
    "description": "Find the country based on SRC.City and SRC.`ZIP Code`."
    "columns_involved": ["City", "ZIP Code"],
}

NOTE: consider the question very carefully when determining how to get the current target column, as it may cue what the target schema means.

Output your result strictly as a Python dictionary, without any extra formatting, explanations, or text. The output must be directly parseable as a Python dictionary.""",
    "extract_mode": """You are a data scientist working with structured tables.

You will be given:
- A table (schema and sample rows).
- A new column to generate, which is needed to answer a question.

Your job is to decide:
1. Should the values of the new column be extracted row-by-row using language reasoning by LLM?
2. Or, can the values be generated using a single Python function that processes the other column(s)?

Output one of:
- 'rowwise_extraction'
- 'python_code'

Output a JSON object directly without any quotes, explanations, or formatting with the following format:
{
    "type": "python_code/rowwise_extraction"
    "explanation": "Explanation of the computations that need to be done to produce the column."
}

Carefully interpret what the new column expects based on the given question, and prioritize python_code, which utilizes Pandas DataFrame, unless LLM is strictly necessary.""",
    "extract_col": """You are a helpful and knowledgeable data scientist.

You will be provided with:
- A table represented by its schema and rows.
- A column to be added to this table whose values depend on the other columns in the table.

Your goal is to determine the values of the new column for all rows. Ensure you consider **all provided columns together** rather than relying on a single column. For example, a city name may exist in multiple locations, but when paired with its corresponding province or county, ambiguity is reduced.

Output your result strictly as a Python list representing the new column values for all rows, without any extra formatting, explanations, or text. The output must be directly parseable as a Python list.""",
    "reduce_row": """You are a helpful and knowledgeable data scientist.

You will be provided with:
- A user's question.
- A table that combines multiple source of information to answer the question.

Your goal is to produce a SQL code (DuckDB) containing predicates to reduce the rows of target_table. In other words, you need to eliminate irrelevant rows.
However, you should do so carefully, especially when comparing equality (e.g., use "LOWER(name) LIKE LOWER('%john%')").

Extra note:
- DuckDB already understands date columns, so you do not need to wrap such columns with `DATE()`.

Output your result strictly as a SQL code (DuckDB) without any extra formatting, explanations, or text. The output must be directly parseable as a SQL code.""",
}

In [11]:
column_extraction_new_prompt = """You are a helpful data scientist.

You will be provided with:
- A source table called SRC that is represented by its schema and some sample rows.
- A target schema, in which a SQL statement will be executed over to answer a user's question.
- A single current target column, which is a key-value pair from the target schema (e.g., "Delayed Ship Date": {"description": "Actual ship date after the 3-day delay", "type": "DATE"}).

Your task is to decide whether to:
1. Select a column from SRC directly (if it maps cleanly to the target column), or
2. Extract or compute the target column using one or more columns from SRC (e.g., adding a time delta, parsing a field, applying a condition).

When deciding this, you must:
- Carefully analyze both the name and explanation of the current target column.
- Consider how the current column helps the execution of the SQL statement.
- Determine what transformation or logic is required to form this column from the source data.

When using extract_column, always include all columns from SRC that are required to perform the extraction, including:
- Main columns used in transformation.
- Helper/reference columns (e.g., timestamps, ZIP codes).
- Any columns used for conditional logic or filters (e.g., shipment method, status).

Be very careful about column names of SRC; do not exclude underscore symbols or whitespaces in the column names of SRC, as these result in errors.

Output format:

If you can directly select a column:
```python
{
    "operation": "select_column",
    "description": "Select SRC.Restaurant ID.",
    "columns_involved": ["Restaurant ID"],
}
```

If the target column must be derived:
```python
{
    "operation": "extract_column",
    "description": "Find the country based on SRC.City and SRC.`ZIP Code`.",
    "columns_involved": ["City", "ZIP Code"],
}
```

Your output must be a single valid Python dictionary. Do not add any extra text, markdown, or formatting."""

In [9]:
DB_SCHEMA_FINAL = "db_schema_final"

### First

In [ ]:
extract_mode = """You are a data scientist working with structured tables.

You will be given:
- A table (schema and sample rows).
- A new column to generate, which is needed as part of executing a SQL statement.

Your job is to decide:
1. Should the values of the new column be extracted row-by-row using language reasoning by LLM?
2. Or, can the values be generated using a single Python function that processes the other column(s)?

Output one of:
- 'rowwise_extraction'
- 'python_code'

Output a JSON object directly without any quotes, explanations, or formatting with the following format:
{
    "type": "python_code/rowwise_extraction"
    "explanation": "Explanation of the computations that need to be done to produce the column. Consider thoroughly what is mentioned in the SQL statement."
}

Carefully interpret what the new column expects based on the given question, and prioritize python_code, which utilizes Pandas DataFrame, unless LLM is strictly necessary."""

In [ ]:
from processor.conductor_state import ConductorState
from processor.table.representation.abstract_table import AbstractTable
from processor.utils.string_processor import clean_code_string, parse_code_string

import pandas as pd


def compute_target_table(
        ctx: ConductorState,
        sql_script: str,
        base_table: AbstractTable,
        target_schema: dict[str, str],
        num_rows=3,
        input_nodes = [],
    ):
        """
        Projects `base_table`, specifically its schema, to the `target_schema`,
        resulting in `target_table`.
        """
        ctx.logger.info("Computing target table")
        target_table_cols: dict[str, list] = dict()
        extra_input_nodes: list = []
        for col in target_schema:
            ctx.logger.info(f"=> Processing column {col}")
            msg = [
                {
                    "role": "system",
                    "content": column_extraction_new_prompt,
                },
                {
                    "role": "user",
                    "content": f"- Source table: ```{base_table.get_representation(num_rows, 42)}```\n- SQL statement: ```{sql_script}```\n- Target Schema: ```{target_schema}```\n- Target Column: ```{col}: {target_schema[col]}```",
                },
            ]
            operation: dict[str, str] = parse_code_string(ctx.llm.chat(msg))
            ctx.logger.info(f"==> Operation: {operation}")
            if operation["operation"] == "select_column":
                operation_node = ctx.computation_graph.create_node(
                    "Mapped a column directly.",
                    list(base_table[operation["columns_involved"][0]]),
                    input_nodes,
                )
            else:
                ctx.logger.info("WARNING: ENTERING EXTRACT_COLUMN")
                operation_node = extract_column(
                    ctx,
                    sql_script,
                    base_table,
                    operation["columns_involved"],
                    f"{col}: {target_schema[col]}",
                    10,
                    num_rows,
                    input_nodes,
                )
            extra_input_nodes.append(operation_node)
            target_table_cols[col] = operation_node.computation_output
        target_table = type(base_table).merge_columns(target_table_cols)
        return ctx.computation_graph.create_node(
            "Projected columns from base table to target table.",
            target_table,
            input_nodes + extra_input_nodes,
        )


def extract_column(
    ctx: ConductorState,
    sql_script: str,
    base_table: AbstractTable,
    columns_involved: list[str],
    target_column: str,
    row_batch=10,
    num_rows = 3,
    input_nodes: list = [],
):
    ctx.logger.info(f"===> Operation extract_column")
    sql_script = "SELECT "
    for col in columns_involved:
        sql_script += f'"{col}", '
    sql_script = sql_script[:-2] + " FROM base_table;"

    ctx.logger.info(f"===> sql_script: {sql_script}")

    columns_involved_table = ctx.table_store.execute_sql_query(
        sql_script,
        {
            "base_table": base_table,
        },
    )
    unique_columns_involved_table = columns_involved_table.drop_duplicates()


    # Ask LLM whether to use row-wise extraction or Python code
    msg = [
        {
            "role": "system",
            "content": extract_mode,
        },
        {
            "role": "user",
            "content": f"- Table: ```{unique_columns_involved_table.get_representation(num_rows)}```\n- Overall Schema: ```{list(base_table.get_schema())}```\n- New Column: ```{target_column}```\n-SQL statement: ```{sql_script}```",
        },
    ]
    output = ctx.llm.chat(msg).strip()
    ctx.logger.info(f"OUTPUT: {output}")
    extraction_mode: dict[str,str] = parse_code_string(output)
    ctx.logger.info(f"===> extraction_mode: {extraction_mode}")
    actual_values: list[str] = []
    if extraction_mode['type'] == "python_code":
        # Generate code from the LLM
        code_gen_msg = [
            {
                "role": "system",
                "content": base_table_reducer_prompts["python_column_extractor"],
            },
            {
                "role": "user",
                "content": f"- Table: ```{unique_columns_involved_table.get_representation(num_rows)}```\n- Target column: ```{target_column}```\n- User's question:\n```{question}```\n- Computation to do: ```{extraction_mode['explanation']}```",
            },
        ]
        code_str = ctx.llm.chat(code_gen_msg)
        ctx.logger.info(f"Python code to extract: {code_str}")
        code_str = clean_code_string(code_str)
        exec_globals = {}
        exec(code_str, exec_globals)

        # ctx.logger.info(f"exec_globals: {exec_globals}")
        import re
        import pandas as pd
        generated_func = exec_globals.get("generate_column")

        if not generated_func:
            raise ValueError("LLM did not return a valid 'generate_column' function.")

        actual_values = generated_func(columns_involved_table.get_data())
    else:
        rows: list[tuple[int, int]] = []
        for i in range(0, len(unique_columns_involved_table), row_batch):
            rows.append((i, i + row_batch))
        rows[-1] = (rows[-1][0], len(unique_columns_involved_table))

        new_col_values: list[str] = []
        for row in tqdm(rows, desc="Processing column extraction"):
            msg = [
                {
                    "role": "system",
                    "content": base_table_reducer_prompts["extract_col"],
                },
                {
                    "role": "user",
                    "content": f"Table ({row[1]-row[0]} rows): ```{unique_columns_involved_table.get_representation(row[1]-row[0], None, True, row)}```\nOverall Schema: {list(base_table.get_schema())}\nNew Column: `{target_column}`",
                },
            ]
            extracted_values = ctx.llm.chat(msg)
            new_col_values.extend(parse_code_string(extracted_values))

        results_cache: dict[str, str] = dict()
        columns = unique_columns_involved_table.get_schema()
        for idx, row in unique_columns_involved_table.iterrows():
            vals = []
            for col in columns:
                vals.append(row[col])
            key = "_SEP_".join(vals)
            results_cache[key] = new_col_values[idx]

        for idx, row in columns_involved_table.iterrows():
            vals = []
            for col in columns:
                vals.append(row[col])
            key = "_SEP_".join(vals)
            actual_values.append(results_cache[key])
    return ctx.computation_graph.create_node(
        "Extraced column values from existing columns in the base table.",
        actual_values,
        input_nodes,
    )

In [139]:
target_table_node = compute_target_table(
    processor.ctx,
    QUESTION_1,
    processor.ctx.table_store.get_table(DB_SCHEMA_FINAL, "base_table"),
    target_schema,
    2,
    # [join_operations_node],
)
target_table = target_table_node.computation_output
print(target_table)

[2025-04-29 03:17:30] INFO in 2589657512: Computing target table
[2025-04-29 03:17:30] INFO in 2589657512: => Processing column ID
[2025-04-29 03:17:43] INFO in 2589657512: ==> Operation: {'operation': 'select_column', 'description': 'Select SRC.ASN_ID as the unique identifier for each shipment.', 'columns_involved': ['ASN_ID']}
[2025-04-29 03:17:43] INFO in 2589657512: => Processing column Ship_Date
[2025-04-29 03:17:54] INFO in 2589657512: ==> Operation: {'operation': 'select_column', 'description': 'Select SRC.SHIPMENT_DATE.', 'columns_involved': ['SHIPMENT_DATE']}
[2025-04-29 03:17:54] INFO in 2589657512: => Processing column Actual_Ship_Date
[2025-04-29 03:18:09] INFO in 2589657512: ==> Operation: {'operation': 'extract_column', 'description': 'Calculate the actual shipping date after a 3-day delay for UPS ground shipments.', 'columns_involved': ['SHIPMENT_DATE', 'CARRIER']}
[2025-04-29 03:18:09] INFO in 2589657512: WARNING: ENTERING EXTRACT_COLUMN
[2025-04-29 03:18:09] INFO in 25

AttributeError: 'NoneType' object has no attribute 'lower'

In [99]:
target_table.get_schema()

['Shipment ID', 'Scheduled Ship Date', 'Delayed Ship Date', 'Item Count']

In [103]:
target_table.get_data()[["Item Count"]].value_counts()    

Item Count
0             100000
Name: count, dtype: int64

In [64]:
target_table[["Item Count"]].value_counts()

Item Count
0             100000
Name: count, dtype: int64

In [75]:
processor.ctx.table_store.add_table(DB_SCHEMA_AFTER_UNION, "target_table", target_table, True, True)

### Predicate

In [ ]:
def execute_sql_query(
        sql_query: str, tables_involved: dict[str, AbstractTable] = None
    ) -> AbstractTable:
        """
        [EXPERIMENTAL] Executes SQL query

        Args:
            sql_query (str): SQL query to execute
            tables_involved (list[AbstractTable]): OPTIONAL - Specify tables to query over (used by, e.g., PyTableStore)
        """
        import duckdb

        # Create an in-memory DuckDB connection
        conn = duckdb.connect(database=":memory:")

        # Make sure you pass a dictionary of DFTable
        if tables_involved is None or len(tables_involved) == 0:
            raise ValueError(
                "PyTableStore requires `tables_involved` to execute SQL queries."
            )

        if not isinstance(tables_involved[list(tables_involved.keys())[0]], DFTable):
            raise ValueError("Only Pandas DataFrame is supported for now.")

        # Register each DataFrame as a DuckDB view
        for table_id, table in tables_involved.items():
            df = table.get_data().copy()
            for col in df.columns:
                if df[col].dtype == "object":
                    try:
                        df[col] = pd.to_datetime(df[col])
                    except Exception:
                        pass  # Not datetime, ignore
            conn.register(table_id.lower(), df)

        # Run your SQL query (assumed lowercase)
        data = conn.execute(sql_query).fetchdf()
        return DFTable(data)

In [ ]:
from processor.utils.string_processor import parse_sql_string


def apply_predicate_to_target_table(
        ctx: ConductorState,
        target_table: AbstractTable,
        question: str,
        num_rows=3,
        input_nodes = [],
    ):
        """
        Applies a natural-language predicate to target table.
        """
        msg = [
            {"role": "system", "content": base_table_reducer_prompts["reduce_row"]},
            {
                "role": "user",
                "content": f"""- Table: ```{target_table.get_representation(num_rows, 42)}```
- Question: {question}\n- Table Schema (careful with column names, e.g., do not forget whitespaces if any): ```{target_table.get_schema()}```""",
            },
        ]
        llm_output = ctx.llm.chat(msg)
        sql_query = parse_sql_string(llm_output)
        ctx.logger.info(f"SQL Query: {sql_query}")
        final_table = execute_sql_query(
            sql_query, {"target_table": target_table}
        )
        return ctx.computation_graph.create_node(
            f"Applied this predicate to the rows of target table: {sql_query}",
            final_table,
            input_nodes,
        )

In [ ]:
final_table_node = apply_predicate_to_target_table(
    processor.ctx,
    processor.ctx.table_store.get_table(DB_SCHEMA_AFTER_UNION, "base_table"),
    QUESTION_1,
    5,
)
final_table = final_table_node.computation_output
print(final_table)

[2025-04-22 20:03:02] INFO in 1467923155: SQL Query: SELECT COUNT(*) 
FROM target_table 
WHERE LOWER(SHIPPING_METHOD) LIKE LOWER('%UPS%') 
AND DELIVERY_DATE > DATE(DISTRIBUTION_TS) + INTERVAL '3 days';


/tmp/ipykernel_2478959/4078682329.py:31: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[col] = pd.to_datetime(df[col])
/tmp/ipykernel_2478959/4078682329.py:31: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[col] = pd.to_datetime(df[col])
/tmp/ipykernel_2478959/4078682329.py:31: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[col] = pd.to_datetime(df[col])
/tmp/ipykernel_2478959/4078682329.py:31: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a 

BinderException: Binder Error: No function matches the given name and argument types 'lower(DOUBLE)'. You might need to add explicit type casts.
	Candidate functions:
	lower(VARCHAR) -> VARCHAR


LINE 3: WHERE LOWER(SHIPPING_METHOD) LIKE LOWER('%UPS%') 
              ^

In [104]:
target_schema

{'Shipment ID': 'Unique identifier for each shipment',
 'Scheduled Ship Date': 'Original scheduled date for the shipment',
 'Delayed Ship Date': 'Actual ship date after the 3-day delay',
 'Item Count': 'Number of items in the shipment'}

In [76]:
QUESTION_1

'Assuming that all shipments by UPS ground service are late by 3 days, how many items will be impacted?'

In [ ]:
# AND DATE("Delivery Date") > "Ship Date" + INTERVAL 3 day

In [ ]:
x = execute_sql_query(
    """SELECT *
FROM target_table 
WHERE LOWER("Courier Service") LIKE LOWER('%UPS ground%')
AND "Delivery Date" > "Ship Date" + INTERVAL 3 day;""",
    {
        "target_table": processor.ctx.table_store.get_table(DB_SCHEMA_AFTER_UNION, "target_table")
    }
)

/tmp/ipykernel_2195825/4078682329.py:31: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[col] = pd.to_datetime(df[col])


In [51]:
x.data

,Shipment ID,Courier Service,Delivery Date,Ship Date


# Visualization

In [ ]:
from pyvis.network import Network
import networkx as nx

from processor.computation_graph import ComputationGraph


def interactive_network_pyvis(graph: ComputationGraph):
    G = nx.DiGraph()

    for node in graph.nodes:
        # label = f"{node.function_name}()\n{node.class_name or ''}\n{node.computation_description}"
        label = f"{node.function_name}()"
        G.add_node(node.id, label=label)

    for node in graph.nodes:
        for input_node in node.input_nodes:
            G.add_edge(input_node.id, node.id)

    net = Network(
        notebook=True,
        height="600px",
        width="100%",
        directed=True,
        cdn_resources="in_line",
    )
    net.from_nx(G)
    net.show("graph.html")  # Will now render inline in Jupyter

In [ ]:
interactive_network_pyvis(processor.ctx.computation_graph)

In [ ]:
# BACKUP OLD
enhanced_schemas = {
    "JI_ASN_CARRIER": [
        "ShippingNoticeID",
        "OrganizationID",
        "SHIPPER_CARRIER",
        "Shipment_Domain",
        "SHIPMENT_UNIQUE_ID",
        "ASN_Timestamp",
    ],
    "JI_PURCHASE_ORDER_LINE": [
        "ORGANIZATION_ID",
        "PURCHASE_ORDER_ID",
        "PO_LINE_ITEM_ID",
        "DEPARTMENT_KEY",
        "SUPPLIER_ID",
        "LINE_ITEM_KEY",
        "PURCHASE_ORDER_NUMBER",
        "CONTRACT_REFERENCE_ID",
        "CONTRACT_ID_NUMBER",
        "ORDER_QUANTITY",
        "TOTAL_LINE_COST",
        "ORDER_CREATION_TIMESTAMP",
        "DISTRIBUTION_SHIPMENT_TS",
        "EXPORT_TIMESTAMP",
        "LAST_REVISION_TIMESTAMP",
        "ORIGINAL_REVISION_TIMESTAMP",
        "WORKFLOW_COMPLETION_TS",
        "ACCOUNTING_DATE_TIMESTAMP",
        "USER_OWNER_ID",
        "USER_SUBMITTER_ID",
        "EXTERNAL_PURCHASE_ORDER_LINE_ID",
        "PO_LINE_NUMBER",
        "UNIT_PRICE_PER_ITEM",
        "CONTRACT_UNIT_PRICE_CONTRACTED",
        "SUPPLIER_ACCOUNTING_CODE",
        "SHIPMENT_METHOD",
        "IS_PO_LINE_AWARDED_BID_FLAG",
        "IS_PO_LINE_REJECTED_FLAG",
        "IS_PO_LINE_CANCELLED_FLAG",
        "IS_LINE_SENT_TO_SUPPLIER",
        "HAS_INVOICES",
        "IS_FORCE_MATCHED",
        "IS_PO_LINE_FORCED_MATCHED",
        "REQUISITION_REQUEST_ID",
        "REQUISITION_IDENTIFIER",
        "REQUISITION_LINE_NUMBER",
        "REQUISITION_LINE_ID",
        "REQUISITION_CREATION_TS",
        "PO_LATEST_REVISION_NUMBER",
        "HAS_REJECTED_RECEIPTS",
        "HAS_CREDITS",
        "IS_OVERSHIPPED",
        "IS_PO_LINE_EXCESS_RECEIPT",
        "IS_PO_LINE_OVERINVOICED",
        "HasSubstitutedInvoiceItems",
        "HAS_CANCELLED_RECEIPT_ITEMS",
        "IS_PO_LINE_HAS_CANCELLED_ITEMS",
        "HAS_RECEIVED_SHIPMENTS",
        "HAS_RETURN_RECEIPTS",
        "HAS_INVOICES",
        "UNIT_PRICE_IN_USD",
        "EXTENDED_PRICE_IN_USD",
        "CONTRACT_UNIT_PRICE_IN_USD",
        "SUPPLIER_RANKING",
        "IS_DIVERSE_SUPPLIER",
        "LIST_PRICE_SET_KEY",
        "LIST_PRICE_SET_DESCRIPTION",
        "CONTRACT_LIST_PRICE\n\nThis_name_better_reflects_that_the_value_in_this_column_is_likely_the_list_price_associated_with_the_contract_for_the_item,_which_is_specific_to_the_context_of_the_purchase_order_and_its_line_items._It_also_maintains_clarity_and_consistency_with_other_column_names_that_describe_pricing_aspects.",
        "LIST_PRICE_SET_VERSION_NUMBER",
        "PREVIOUS_LIST_PRICE_VERSION",
        "PREVIOUS_LIST_PRICE_SET_VERSION_NAME",
        "Purchase_Order_Type_Code",
        "Purchase_Order_Type",
        "RECEIPT_STATUS_ENUM",
        "RECEIPT_STATUS",
        "PO_Invoice_Status_Type",
        "INVOICE_STATUS",
        "PO_Workflow_Status_Type",
        "Purchase_Order_Workflow_Status",
        "PO_Match_Status_Enum",
        "PURCHASE_ORDER_MATCH_STATUS",
        "UNIT_PRICE_SOURCE_TYPE",
        "UNIT_PRICE_SOURCE_DESCRIPTION",
        "PO_LINE_MATCH_STATUS_DESCRIPTION",
        "PO_LINE_MATCHING_STATUS",
        "Contract_Unit_Price_Business",
        "CONTRACT_UNIT_PRICE_BUSINESS_CURRENCY_CODE",
        "CONTRACT_UNIT_PRICE_BUSINESS_EXCHANGE_RATE_DESCRIPTION",
        "EXTENDED_PRICE_BUSINESS_VALUE",
        "BUSINESS_EXTENDED_PRICE_CURRENCY",
        "EXTENDED_PRICE_BUSINESS_EXCHANGE_RATE_VALUE",
        "BUSINESS_UNIT_PRICE",
        "BUSINESS_UNIT_PRICE_CURRENCY",
        "EXCHANGE_RATE_BUSINESS_TO_USD",
        "HAS_SHIPPED",
        "HAS_RECEIVED_SHIPMENTS",
        "PO_LINE_RECEIPT_STATUS_DESCRIPTION\n\nThis_name_provides_clarity_about_the_nature_of_the_data_stored_in_the_column,_indicating_that_it_describes_the_status_of_receipts_for_each_PO_line.",
        "RECEIPT_STATUS_ENUM",
        "PO_LINE_SHIPMENT_STATUS_DESCRIPTION",
        "SHIPMENT_STATUS",
        "MAX_UNIT_PRICE_RECEIVED",
        "MINIMUM_RECEIPT_UNIT_PRICE",
        "HAS_CANCELLED_RECEIPT_ITEMS",
        "IS_PO_LINE_REQUIRES_RECEIPT_MATCHING",
        "IS_PO_LINE_MATCHING_REQUIRES_RECEIPT",
        "IS_PO_LINE_SHIPPED_IN_EXCESS",
        "MAX_RECEIPT_UNIT_PRICE_USD",
        "MIN_RECEIPT_UNIT_PRICE_USD",
        "SHIP_TO_ADDRESS_IDENTIFIER",
        "Bill_To_Address_ID",
        "COMMODITY_CODE_KEY",
        "REQUESTED_DELIVERY_DATE",
        "DELIVERY_DATE_CATEGORY",
        "DELIVERY_LEAD_TIME_IN_DAYS",
        "FORM_DOCUMENT_ID",
        "FORM_REQUEST_ID",
        "TOTAL_PURCHASE_ORDER_AMOUNT",
        "TOTAL_DOCUMENT_AMOUNT",
        "Document_Currency_Type",
        "Exchange_Rate_Grand_Total_Document",
        "Total_USD",
        "LAST_TRANSFORM_TS",
        "FULFILLMENT_CENTER_ID",
        "TOP_LEVEL_CATEGORY_NAME",
        "UNSPSC_Category_Level_1",
        "CATEGORY_LEVEL_2_DESCRIPTION",
        "UNSPSC_Category_Level_2",
    ],
    "JI_ORDER_ACK_LINE": [
        "Order_Acknowledgment_ID",
        "ShipmentLineIdentifier",
        "PURCHASE_ORDER_LINE_ID",
        "OrganizationID",
        "ORDER_QUANTITY",
        "EstimatedShippingDate",
        "Order_Status_Code",
        "Order_Ack_Status",
        "ACKNOWLEDGEMENT_NOTES",
        "LAST_UPDATED_TS",
    ],
    "JI_ASN_LINE": [
        "ASN_Line_ID",
        "ASN_LINE_ID",
        "PURCHASE_ORDER_LINE_ID",
        "Organization_ID",
        "Shipped_Quantity",
        "SHIPMENT_NOTES",
        "SHIPMENT_RECORD_TIMESTAMP",
    ],
    "JI_ASN": [
        "AdvancedShippingNoticeID",
        "Shipment_ID",
        "Originating_Organization_ID",
        "ScheduledShipmentDate",
        "DELIVERY_DATE",
        "SHIPMENT_COMMENTS",
        "LAST_UPDATE_TS",
    ],
}